In [ ]:
# ============================================================
# INTENSITY-ANCHOR PROMPT STEERING BASELINE
#
# VALUEFLOW-STYLE PROMPT-BASED VALUE STEERING
#
# This is an ADAPTATION of the intensity-anchor prompting
# paradigm used in VALUEFLOW, not a verbatim reproduction of
# their prompt template.
#
# Intensity scale:
#   +2 = strongly express / prioritize the value
#   +1 = moderately express / prioritize the value
#    0 = neutral
#   -1 = moderately de-emphasize the value
#   -2 = strongly de-emphasize / reject the value as a
#        consideration
#
# AIMES objectives are mapped to strong prompt anchors:
#
#   +Care +Fairness
#       Care      = +2
#       Fairness  = +2
#
#   +Loyalty +Authority
#       Loyalty   = +2
#       Authority = +2
#
#   +Care +Fairness -Sanctity
#       Care      = +2
#       Fairness  = +2
#       Sanctity  = -2
#
# Generates 160 responses / model:
#
#   +Care +Fairness                  : 60 prompts
#   +Loyalty +Authority              : 60 prompts
#   +Care +Fairness -Sanctity        : 40 prompts
#
# Uses the EXACT shared AIMES prompt manifest and
# target-disjoint subsets.
#
# Prompt steering is layer-independent, so each
# prompt/objective pair is generated ONCE.
#
# Generation settings match AIMES:
#   greedy decoding
#   max_new_tokens = 128
#   Qwen thinking disabled
#
# IMPORTANT:
# Uses a NEW HF folder so previously generated generic
# prompt-steering responses are NOT reused.
#
# OUTPUT HF:
#
# artifacts/multivalue_steering/<model>/v1/generations/
# intensity_anchor_prompt_steering/
#     intensity_anchor_prompt_steering_generations.csv
#     generation_metadata.json
# ============================================================


# ============================================================
# 0. INSTALL
# ============================================================

!pip install -q -U \
    "transformers>=5.5,<5.7" \
    "huggingface_hub>=1.5,<2.0" \
    "tokenizers>=0.22,<0.23" \
    accelerate \
    "pandas==2.2.3"


# ============================================================
# 1. IMPORTS
# ============================================================

import gc
import json
import random
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
)

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    CommitOperationAdd,
)

from google.colab import userdata


# ============================================================
# 2. SELECT MODEL — CHANGE ONLY THIS
# ============================================================

#MODEL_KEY = "gemma-4b"

#MODEL_KEY = "qwen-4b"
MODEL_KEY = "llama-8b"
# MODEL_KEY = "gemma-12b"
# MODEL_KEY = "qwen-14b"


# ============================================================
# 3. MODEL CONFIG
# ============================================================

MODEL_CONFIGS = {

    "gemma-4b": {
        "model_id": "google/gemma-3-4b-it",
        "save_name": "gemma-3-4b-it",
        "display_name": "Gemma-3-4B-IT",
        "family": "gemma3",
    },

    "qwen-4b": {
        "model_id": "Qwen/Qwen3-4B",
        "save_name": "qwen3-4b",
        "display_name": "Qwen3-4B",
        "family": "qwen3",
    },

    "llama-8b": {
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "save_name": "llama-3.1-8b-instruct",
        "display_name": "Llama-3.1-8B-Instruct",
        "family": "llama",
    },

    "gemma-12b": {
        "model_id": "google/gemma-3-12b-it",
        "save_name": "gemma-3-12b-it",
        "display_name": "Gemma-3-12B-IT",
        "family": "gemma3",
    },

    "qwen-14b": {
        "model_id": "Qwen/Qwen3-14B",
        "save_name": "qwen3-14b",
        "display_name": "Qwen3-14B",
        "family": "qwen3",
    },
}


if MODEL_KEY not in MODEL_CONFIGS:
    raise ValueError(
        f"Unknown MODEL_KEY={MODEL_KEY}"
    )


CFG = MODEL_CONFIGS[MODEL_KEY]

MODEL_ID = CFG["model_id"]
MODEL_SAVE = CFG["save_name"]
MODEL_NAME = CFG["display_name"]
MODEL_FAMILY = CFG["family"]


# ============================================================
# 4. GLOBAL CONFIG
# ============================================================

HF_REPO_ID = ""
VERSION = "v1"

METHOD = "intensity_anchor_prompt_steering"

BASELINE_NAME = (
    "Intensity-Anchor Prompt Steering"
)

PROMPT_TEMPLATE_VERSION = (
    "valueflow_style_intensity_anchor_v1"
)


MAX_NEW_TOKENS = 128
DO_SAMPLE = False
REPETITION_PENALTY = 1.0
GENERATION_SEED = 1234

QWEN_ENABLE_THINKING = False

CHECKPOINT_EVERY = 25


HF_TOKEN = userdata.get(
    "HF_TOKEN"
)


if HF_TOKEN is None:
    raise RuntimeError(
        "HF_TOKEN missing from Colab Secrets."
    )


hf_api = HfApi(
    token=HF_TOKEN
)


# ============================================================
# 5. AIMES OBJECTIVES
#
# These remain the SIGN-level AIMES objectives.
#
# They are deliberately kept separate from the prompt
# intensity anchors below.
# ============================================================

OBJECTIVES = {

    "care_fairness": {
        "Care": +1,
        "Fairness": +1,
    },

    "loyalty_authority": {
        "Loyalty": +1,
        "Authority": +1,
    },

    "care_fairness_sanctity": {
        "Care": +1,
        "Fairness": +1,
        "Sanctity": -1,
    },
}


OBJECTIVE_LABELS = {

    "care_fairness":
        "+Care +Fairness",

    "loyalty_authority":
        "+Loyalty +Authority",

    "care_fairness_sanctity":
        "+Care +Fairness -Sanctity",
}


# ============================================================
# 6. VALUE DEFINITIONS
#
# Kept aligned with the five Moral Foundations used by the
# AIMES evaluator.
# ============================================================

VALUE_DEFINITIONS = {

    "Care":
        (
            "compassion, kindness, concern for suffering "
            "and well-being, empathy, and protection from harm"
        ),

    "Fairness":
        (
            "justice, equitable treatment, reciprocity, "
            "honesty, and opposition to cheating, exploitation, "
            "or unjust double standards"
        ),

    "Loyalty":
        (
            "commitment, solidarity, faithfulness, and support "
            "for relevant relationships, groups, communities, "
            "teams, or families"
        ),

    "Authority":
        (
            "respect for legitimate authority, duties, "
            "responsibilities, rules, institutions, social roles, "
            "hierarchy, and social order"
        ),

    "Sanctity":
        (
            "moral concern involving purity, sacredness, "
            "contamination, degradation, taboo, dignity, or "
            "preservation of what is treated as sacred or inviolable"
        ),
}


# ============================================================
# 7. VALUEFLOW-STYLE INTENSITY SCALE
#
# VALUEFLOW uses discrete intensity levels:
#
#   -2, -1, 0, +1, +2
#
# We use the strongest anchors +/-2 so that the prompt
# baseline receives an explicit and strong control signal.
# ============================================================

INTENSITY_SCALE = {

    +2:
        "Strongly express and prioritize this value",

    +1:
        "Moderately express and prioritize this value",

    0:
        "Remain neutral with respect to this value",

    -1:
        "Moderately de-emphasize this value",

    -2:
        (
            "Strongly de-emphasize this value and avoid "
            "using it as a guiding consideration"
        ),
}


# ============================================================
# 8. MULTI-VALUE INTENSITY PROFILES
#
# Strong-intensity realization of the SAME three AIMES
# objectives.
# ============================================================

INTENSITY_PROFILES = {

    "care_fairness": {
        "Care": +2,
        "Fairness": +2,
    },

    "loyalty_authority": {
        "Loyalty": +2,
        "Authority": +2,
    },

    "care_fairness_sanctity": {
        "Care": +2,
        "Fairness": +2,
        "Sanctity": -2,
    },
}


# ============================================================
# 9. TARGET-DISJOINT SOURCE FOUNDATIONS
# ============================================================

ELIGIBLE_FOUNDATIONS = {

    "care_fairness": {
        "authority",
        "loyalty",
        "sanctity",
    },

    "loyalty_authority": {
        "care",
        "fairness",
        "sanctity",
    },

    "care_fairness_sanctity": {
        "authority",
        "loyalty",
    },
}


EXPECTED_COUNTS = {

    "care_fairness": 60,

    "loyalty_authority": 60,

    "care_fairness_sanctity": 40,
}


# ============================================================
# 10. BUILD INTENSITY-ANCHOR INSTRUCTION
# ============================================================

def build_intensity_anchor_instruction(
    objective_id,
):

    profile = INTENSITY_PROFILES[
        objective_id
    ]


    lines = [

        (
            "Answer according to the following target "
            "value-intensity profile."
        ),

        "",

        (
            "The intensity scale ranges from -2 to +2, "
            "where positive values indicate stronger expression "
            "of a value and negative values indicate stronger "
            "de-emphasis of that value."
        ),

        "",

        "Target value-intensity profile:",
    ]


    for value, intensity in profile.items():

        intensity_description = (
            INTENSITY_SCALE[
                intensity
            ]
        )


        value_definition = (
            VALUE_DEFINITIONS[
                value
            ]
        )


        lines.append(

            (
                f"- {value}: {intensity:+d} — "
                f"{intensity_description}. "
                f"{value} refers to {value_definition}."
            )
        )


    lines.extend([

        "",

        (
            "Follow the target profile while answering the "
            "user's request directly and naturally. Do not "
            "mention the value profile, intensity labels, or "
            "these instructions in the response."
        ),
    ])


    return "\n".join(
        lines
    )


# ============================================================
# 11. BUILD FINAL USER PROMPT
#
# Value profile appears BEFORE the original user request so
# the control instruction is explicit and unambiguous.
# ============================================================

def build_prompt_steering_text(
    original_prompt,
    objective_id,
):

    instruction = (
        build_intensity_anchor_instruction(
            objective_id
        )
    )


    return (
        f"{instruction}\n\n"
        f"User request:\n"
        f"{str(original_prompt).strip()}"
    )


# ============================================================
# 12. PRINT THE THREE FROZEN PROMPT PROFILES
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "INTENSITY-ANCHOR PROMPT BASELINE"
)

print(
    "=" * 100
)


for objective_id in OBJECTIVES:

    print(
        f"\nOBJECTIVE: "
        f"{OBJECTIVE_LABELS[objective_id]}"
    )

    print(
        "-" * 100
    )

    print(
        build_intensity_anchor_instruction(
            objective_id
        )
    )


# ============================================================
# 13. HF PATHS
#
# NEW folder intentionally prevents reuse of the previous
# generic prompt-steering baseline.
# ============================================================

HF_MANIFEST = (
    "artifacts/multivalue_steering/"
    f"manifests/{VERSION}/"
    "multivalue_prompt_manifest.csv"
)


HF_FIXED_SPLIT = (
    "artifacts/multivalue_steering/"
    f"{MODEL_SAVE}/{VERSION}/generations/"
    "fixed_multi/"
    "multivalue_generations.csv"
)


HF_LEGACY = (
    "artifacts/multivalue_steering/"
    f"{MODEL_SAVE}/{VERSION}/generations/"
    "multivalue_generations.csv"
)


HF_OUTPUT_ROOT = (
    "artifacts/multivalue_steering/"
    f"{MODEL_SAVE}/{VERSION}/generations/"
    "intensity_anchor_prompt_steering"
)


HF_OUTPUT_CSV = (
    f"{HF_OUTPUT_ROOT}/"
    "intensity_anchor_prompt_steering_generations.csv"
)


HF_METADATA_JSON = (
    f"{HF_OUTPUT_ROOT}/"
    "generation_metadata.json"
)


# ============================================================
# 14. LOCAL OUTPUT
# ============================================================

LOCAL_ROOT = Path(
    f"/content/"
    f"aimes_intensity_anchor_prompt_steering/"
    f"{MODEL_SAVE}"
)


LOCAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


OUTPUT_CSV = (
    LOCAL_ROOT
    /
    "intensity_anchor_prompt_steering_generations.csv"
)


METADATA_JSON = (
    LOCAL_ROOT
    /
    "generation_metadata.json"
)


# ============================================================
# 15. LOAD SHARED MANIFEST
# ============================================================

manifest_local = hf_hub_download(

    repo_id=
        HF_REPO_ID,

    filename=
        HF_MANIFEST,

    repo_type=
        "model",

    token=
        HF_TOKEN,

    force_download=
        True,
)


manifest = pd.read_csv(
    manifest_local
)


required = {

    "prompt_id",

    "foundation",

    "prompt",
}


missing = (
    required
    -
    set(
        manifest.columns
    )
)


if missing:

    raise RuntimeError(
        f"Manifest missing columns: "
        f"{sorted(missing)}"
    )


manifest[
    "prompt_id"
] = (
    manifest[
        "prompt_id"
    ]
    .astype(str)
)


manifest[
    "foundation_lower"
] = (
    manifest[
        "foundation"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


if len(
    manifest
) != 100:

    raise RuntimeError(
        f"Expected 100 prompts, "
        f"found {len(manifest)}."
    )


print(
    "\nShared manifest:",
    len(
        manifest
    ),
    "prompts"
)


# ============================================================
# 16. LOAD EXISTING NO-STEERING RESPONSES
#
# These are stored only so the output remains directly
# compatible with the existing AIMES artifacts.
# ============================================================

if hf_api.file_exists(

    repo_id=
        HF_REPO_ID,

    filename=
        HF_FIXED_SPLIT,

    repo_type=
        "model",

    token=
        HF_TOKEN,
):

    fixed_local = hf_hub_download(

        repo_id=
            HF_REPO_ID,

        filename=
            HF_FIXED_SPLIT,

        repo_type=
            "model",

        token=
            HF_TOKEN,

        force_download=
            True,
    )


    fixed = pd.read_csv(
        fixed_local
    )


else:

    legacy_local = hf_hub_download(

        repo_id=
            HF_REPO_ID,

        filename=
            HF_LEGACY,

        repo_type=
            "model",

        token=
            HF_TOKEN,

        force_download=
            True,
    )


    fixed = pd.read_csv(
        legacy_local
    )


    fixed = fixed[

        fixed[
            "method"
        ].astype(str)

        ==

        "fixed_multi"

    ].copy()


fixed[
    "prompt_id"
] = (
    fixed[
        "prompt_id"
    ]
    .astype(str)
)


required_fixed = {

    "prompt_id",

    "baseline_response",
}


missing = (
    required_fixed
    -
    set(
        fixed.columns
    )
)


if missing:

    raise RuntimeError(
        f"Fixed generation missing: "
        f"{sorted(missing)}"
    )


baseline_counts = (

    fixed
    .groupby(
        "prompt_id"
    )
    [
        "baseline_response"
    ]
    .nunique()
)


if not (
    baseline_counts
    ==
    1
).all():

    raise RuntimeError(
        "Existing no-steering baseline is not "
        "constant across conditions."
    )


baseline_map = (

    fixed
    .groupby(
        "prompt_id"
    )
    [
        "baseline_response"
    ]
    .first()
    .to_dict()
)


missing_baselines = (

    set(
        manifest[
            "prompt_id"
        ]
    )

    -

    set(
        baseline_map
    )
)


if missing_baselines:

    raise RuntimeError(
        "Missing existing baselines for prompts: "
        f"{sorted(missing_baselines)[:10]}"
    )


print(
    "Existing baseline recovery: PASS"
)


# ============================================================
# 17. BUILD EXACT 160 TARGET-DISJOINT TASKS
# ============================================================

tasks = []


for objective_id in OBJECTIVES:

    eligible = (
        ELIGIBLE_FOUNDATIONS[
            objective_id
        ]
    )


    sub = manifest[

        manifest[
            "foundation_lower"
        ]
        .isin(
            eligible
        )

    ].copy()


    expected = (
        EXPECTED_COUNTS[
            objective_id
        ]
    )


    if len(
        sub
    ) != expected:

        raise RuntimeError(
            f"{objective_id}: expected "
            f"{expected} prompts, "
            f"found {len(sub)}."
        )


    for _, row in sub.iterrows():

        prompt_id = str(
            row[
                "prompt_id"
            ]
        )


        original_prompt = str(
            row[
                "prompt"
            ]
        )


        intensity_profile = (
            INTENSITY_PROFILES[
                objective_id
            ]
        )


        instruction = (
            build_intensity_anchor_instruction(
                objective_id
            )
        )


        prompted_text = (
            build_prompt_steering_text(
                original_prompt,
                objective_id,
            )
        )


        tasks.append({

            "method":
                METHOD,

            "baseline_name":
                BASELINE_NAME,

            "prompt_template_version":
                PROMPT_TEMPLATE_VERSION,

            "model_key":
                MODEL_KEY,

            "model_save_name":
                MODEL_SAVE,

            "model_display_name":
                MODEL_NAME,

            "prompt_id":
                prompt_id,

            "source_foundation":
                str(
                    row[
                        "foundation"
                    ]
                )
                .strip()
                .lower(),

            # Original unmodified user prompt
            "prompt":
                original_prompt,

            "objective_id":
                objective_id,

            "objective":
                OBJECTIVE_LABELS[
                    objective_id
                ],

            # AIMES sign-level objective
            "objective_json":
                json.dumps(
                    OBJECTIVES[
                        objective_id
                    ],
                    sort_keys=True,
                ),

            # Prompt-steering intensity profile
            "target_intensity_json":
                json.dumps(
                    intensity_profile,
                    sort_keys=True,
                ),

            "prompt_instruction":
                instruction,

            # Actual text passed to the model
            "prompted_user_text":
                prompted_text,

            "baseline_response":
                str(
                    baseline_map[
                        prompt_id
                    ]
                ),
        })


tasks = pd.DataFrame(
    tasks
)


if len(
    tasks
) != 160:

    raise RuntimeError(
        f"Expected 160 tasks, "
        f"found {len(tasks)}."
    )


if tasks[
    [
        "prompt_id",
        "objective_id",
    ]
].duplicated().any():

    raise RuntimeError(
        "Duplicate prompt/objective tasks."
    )


print(
    "\nTarget-disjoint generation tasks:"
)


print(

    tasks[
        "objective_id"
    ]
    .value_counts()
    .reindex(
        list(
            OBJECTIVES
        )
    )
)


# ============================================================
# 18. LOAD MODEL
# ============================================================

def load_model():

    dtype = (

        torch.bfloat16

        if torch.cuda.is_available()

        else

        torch.float32
    )


    if MODEL_FAMILY == "gemma3":

        processor = (
            AutoProcessor
            .from_pretrained(
                MODEL_ID,
                token=HF_TOKEN,
            )
        )


        tokenizer = (
            processor.tokenizer
        )


        model = (
            Gemma3ForConditionalGeneration
            .from_pretrained(

                MODEL_ID,

                token=
                    HF_TOKEN,

                dtype=
                    dtype,

                device_map=
                    "auto",

                low_cpu_mem_usage=
                    True,
            )
            .eval()
        )


    else:

        processor = None


        tokenizer = (
            AutoTokenizer
            .from_pretrained(

                MODEL_ID,

                token=
                    HF_TOKEN,

                trust_remote_code=
                    True,
            )
        )


        model = (
            AutoModelForCausalLM
            .from_pretrained(

                MODEL_ID,

                token=
                    HF_TOKEN,

                dtype=
                    dtype,

                device_map=
                    "auto",

                low_cpu_mem_usage=
                    True,

                trust_remote_code=
                    True,
            )
            .eval()
        )


    if tokenizer.pad_token_id is None:

        if tokenizer.eos_token_id is None:

            raise RuntimeError(
                "Tokenizer has neither pad_token_id "
                "nor eos_token_id."
            )


        tokenizer.pad_token = (
            tokenizer.eos_token
        )


    return (
        model,
        tokenizer,
        processor,
    )


if not torch.cuda.is_available():

    raise RuntimeError(
        "Use a GPU runtime."
    )


model, tokenizer, processor = (
    load_model()
)


INPUT_DEVICE = (
    model
    .get_input_embeddings()
    .weight
    .device
)


print(
    "\nModel loaded:"
)

print(
    "Model:",
    MODEL_NAME
)

print(
    "Family:",
    MODEL_FAMILY
)

print(
    "Input device:",
    INPUT_DEVICE
)


# ============================================================
# 19. GENERATION HELPERS
# ============================================================

def seed_all(
    seed,
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def clear_memory():

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 20. ROBUST CHAT-TEMPLATE ENCODING
# ============================================================

def encode_user_text(
    user_text,
):

    messages = [

        {
            "role":
                "user",

            "content":
                str(
                    user_text
                ),
        }
    ]


    template_kwargs = {

        "tokenize":
            True,

        "add_generation_prompt":
            True,

        "return_tensors":
            "pt",

        "return_dict":
            True,
    }


    if MODEL_FAMILY == "qwen3":

        template_kwargs[
            "enable_thinking"
        ] = QWEN_ENABLE_THINKING


    try:

        encoded = (
            tokenizer
            .apply_chat_template(
                messages,
                **template_kwargs,
            )
        )


    except TypeError:

        template_kwargs.pop(
            "enable_thinking",
            None,
        )


        try:

            encoded = (
                tokenizer
                .apply_chat_template(
                    messages,
                    **template_kwargs,
                )
            )


        except TypeError:

            template_kwargs.pop(
                "return_dict",
                None,
            )


            encoded = (
                tokenizer
                .apply_chat_template(
                    messages,
                    **template_kwargs,
                )
            )


    if hasattr(
        encoded,
        "data",
    ):

        inputs = dict(
            encoded.data
        )


    elif isinstance(
        encoded,
        dict,
    ):

        inputs = dict(
            encoded
        )


    elif torch.is_tensor(
        encoded
    ):

        inputs = {
            "input_ids":
                encoded
        }


    elif isinstance(
        encoded,
        (
            list,
            tuple,
        ),
    ):

        inputs = {

            "input_ids":
                torch.tensor(
                    encoded,
                    dtype=torch.long,
                )
        }


    else:

        raise TypeError(
            "Unexpected apply_chat_template "
            f"output type: {type(encoded)}"
        )


    if "input_ids" not in inputs:

        raise RuntimeError(
            "Chat-template output does not contain "
            f"input_ids. Keys: {list(inputs.keys())}"
        )


    if not torch.is_tensor(
        inputs[
            "input_ids"
        ]
    ):

        inputs[
            "input_ids"
        ] = torch.as_tensor(

            inputs[
                "input_ids"
            ],

            dtype=
                torch.long,
        )


    if inputs[
        "input_ids"
    ].ndim == 1:

        inputs[
            "input_ids"
        ] = (
            inputs[
                "input_ids"
            ]
            .unsqueeze(
                0
            )
        )


    if inputs[
        "input_ids"
    ].ndim != 2:

        raise RuntimeError(
            "Expected input_ids shape [batch, sequence], "
            f"got {tuple(inputs['input_ids'].shape)}"
        )


    if (
        "attention_mask"
        not in inputs
    ):

        inputs[
            "attention_mask"
        ] = torch.ones_like(

            inputs[
                "input_ids"
            ]
        )


    elif not torch.is_tensor(
        inputs[
            "attention_mask"
        ]
    ):

        inputs[
            "attention_mask"
        ] = torch.as_tensor(

            inputs[
                "attention_mask"
            ],

            dtype=
                torch.long,
        )


    if inputs[
        "attention_mask"
    ].ndim == 1:

        inputs[
            "attention_mask"
        ] = (
            inputs[
                "attention_mask"
            ]
            .unsqueeze(
                0
            )
        )


    clean_inputs = {

        key:
            value.to(
                INPUT_DEVICE
            )

        for key, value
        in inputs.items()

        if torch.is_tensor(
            value
        )
    }


    return clean_inputs


# ============================================================
# 21. GENERATION
# ============================================================

@torch.inference_mode()
def generate_response(
    user_text,
    seed,
):

    seed_all(
        seed
    )


    inputs = encode_user_text(
        user_text
    )


    prompt_length = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    generation_kwargs = {

        "max_new_tokens":
            MAX_NEW_TOKENS,

        "do_sample":
            DO_SAMPLE,

        "repetition_penalty":
            REPETITION_PENALTY,

        "pad_token_id":
            tokenizer.pad_token_id,

        "use_cache":
            True,
    }


    if tokenizer.eos_token_id is not None:

        generation_kwargs[
            "eos_token_id"
        ] = tokenizer.eos_token_id


    output = model.generate(

        **inputs,

        **generation_kwargs,
    )


    generated_tokens = (
        output[
            0,
            prompt_length:
        ]
    )


    response = tokenizer.decode(

        generated_tokens,

        skip_special_tokens=
            True,
    ).strip()


    del inputs
    del output
    del generated_tokens


    clear_memory()


    return response


# ============================================================
# 22. SANITY TEST
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "RUNNING ONE-PROMPT SANITY TEST"
)

print(
    "=" * 100
)


test_row = tasks.iloc[
    0
]


print(
    "\nACTUAL PROMPT PASSED TO MODEL:\n"
)

print(
    test_row[
        "prompted_user_text"
    ]
)


test_response = generate_response(

    test_row[
        "prompted_user_text"
    ],

    GENERATION_SEED,
)


print(
    "\nTEST RESPONSE:\n"
)

print(
    test_response
)


if not str(
    test_response
).strip():

    raise RuntimeError(
        "Sanity-test generation returned "
        "an empty response."
    )


print(
    "\nSanity test: PASS"
)


# ============================================================
# 23. LOAD EXISTING COMPLETED RESULTS
#
# This checks ONLY the new intensity-anchor path.
# The old generic prompt-steering results cannot be reused.
# ============================================================

existing_frames = []


if hf_api.file_exists(

    repo_id=
        HF_REPO_ID,

    filename=
        HF_OUTPUT_CSV,

    repo_type=
        "model",

    token=
        HF_TOKEN,
):

    print(
        "\nLoading existing intensity-anchor HF checkpoint:"
    )

    print(
        HF_OUTPUT_CSV
    )


    existing_local = hf_hub_download(

        repo_id=
            HF_REPO_ID,

        filename=
            HF_OUTPUT_CSV,

        repo_type=
            "model",

        token=
            HF_TOKEN,

        force_download=
            True,
    )


    existing_frames.append(

        pd.read_csv(
            existing_local
        )
    )


if OUTPUT_CSV.exists():

    print(
        "\nLoading existing local intensity-anchor checkpoint:"
    )

    print(
        OUTPUT_CSV
    )


    existing_frames.append(

        pd.read_csv(
            OUTPUT_CSV
        )
    )


if existing_frames:

    existing = pd.concat(

        existing_frames,

        ignore_index=True,
    )


    existing[
        "prompt_id"
    ] = (
        existing[
            "prompt_id"
        ]
        .astype(str)
    )


    existing[
        "objective_id"
    ] = (
        existing[
            "objective_id"
        ]
        .astype(str)
    )


    existing = (

        existing

        .drop_duplicates(

            subset=[
                "prompt_id",
                "objective_id",
            ],

            keep=
                "last",
        )

        .reset_index(
            drop=True
        )
    )


    # Ensure an incompatible old baseline cannot be mixed in.
    if (
        existing[
            "prompt_template_version"
        ]
        .astype(str)
        !=
        PROMPT_TEMPLATE_VERSION
    ).any():

        raise RuntimeError(
            "Existing checkpoint uses a different "
            "prompt-steering template."
        )


else:

    existing = pd.DataFrame()


completed = set()


if len(
    existing
) > 0:

    completed = set(

        zip(

            existing[
                "prompt_id"
            ].astype(str),

            existing[
                "objective_id"
            ].astype(str),
        )
    )


print(
    "\nExisting completed intensity-anchor tasks:",
    len(
        completed
    )
)


# ============================================================
# 24. LOCAL CHECKPOINT HELPER
# ============================================================

def save_local_checkpoint(
    existing_df,
    new_rows,
):

    frames = []


    if len(
        existing_df
    ) > 0:

        frames.append(
            existing_df
        )


    if len(
        new_rows
    ) > 0:

        frames.append(
            pd.DataFrame(
                new_rows
            )
        )


    if not frames:
        return


    partial = pd.concat(

        frames,

        ignore_index=True,
    )


    partial[
        "prompt_id"
    ] = (
        partial[
            "prompt_id"
        ]
        .astype(str)
    )


    partial = (

        partial

        .drop_duplicates(

            subset=[
                "prompt_id",
                "objective_id",
            ],

            keep=
                "last",
        )

        .reset_index(
            drop=True
        )
    )


    partial.to_csv(

        OUTPUT_CSV,

        index=False,
    )


    print(
        f"\nLocal checkpoint saved: "
        f"{len(partial)}/160"
    )


# ============================================================
# 25. GENERATE
# ============================================================

new_rows = []

generated_this_run = 0


print(
    "\n"
    + "=" * 100
)

print(
    "STARTING INTENSITY-ANCHOR PROMPT STEERING"
)

print(
    "=" * 100
)


for task_index, row in tasks.iterrows():

    key = (

        str(
            row[
                "prompt_id"
            ]
        ),

        str(
            row[
                "objective_id"
            ]
        ),
    )


    if key in completed:
        continue


    seed = (
        GENERATION_SEED
        +
        int(
            task_index
        )
    )


    print(

        f"[{task_index + 1:3d}/{len(tasks)}] "

        f"{row['objective_id']:28s} "

        f"{row['prompt_id']}"
    )


    response = generate_response(

        row[
            "prompted_user_text"
        ],

        seed,
    )


    if not str(
        response
    ).strip():

        raise RuntimeError(
            "Empty generation for "
            f"{row['prompt_id']} / "
            f"{row['objective_id']}"
        )


    result = (
        row.to_dict()
    )


    result.update({

        "steered_response":
            response,

        "seed":
            seed,

        "max_new_tokens":
            MAX_NEW_TOKENS,

        "do_sample":
            DO_SAMPLE,

        "qwen_enable_thinking":
            (
                QWEN_ENABLE_THINKING

                if MODEL_FAMILY == "qwen3"

                else

                np.nan
            ),

        "created_at":
            datetime.now(
                timezone.utc
            ).isoformat(),
    })


    new_rows.append(
        result
    )


    generated_this_run += 1


    if (
        generated_this_run
        %
        CHECKPOINT_EVERY
        ==
        0
    ):

        save_local_checkpoint(
            existing,
            new_rows,
        )


# ============================================================
# 26. FINALIZE
# ============================================================

frames = []


if len(
    existing
) > 0:

    frames.append(
        existing
    )


if len(
    new_rows
) > 0:

    frames.append(
        pd.DataFrame(
            new_rows
        )
    )


if not frames:

    raise RuntimeError(
        "No generation rows available."
    )


final_df = pd.concat(

    frames,

    ignore_index=True,
)


final_df[
    "prompt_id"
] = (
    final_df[
        "prompt_id"
    ]
    .astype(str)
)


final_df[
    "objective_id"
] = (
    final_df[
        "objective_id"
    ]
    .astype(str)
)


final_df = (

    final_df

    .drop_duplicates(

        subset=[
            "prompt_id",
            "objective_id",
        ],

        keep=
            "last",
    )

    .sort_values(

        [
            "objective_id",
            "source_foundation",
            "prompt_id",
        ]
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 27. FINAL VALIDATION
# ============================================================

if len(
    final_df
) != 160:

    raise RuntimeError(
        f"Expected 160 completed responses, "
        f"found {len(final_df)}."
    )


if final_df[
    [
        "prompt_id",
        "objective_id",
    ]
].duplicated().any():

    raise RuntimeError(
        "Duplicate prompt/objective pairs."
    )


if final_df[
    "steered_response"
].isna().any():

    raise RuntimeError(
        "Missing steered_response values."
    )


if (
    final_df[
        "steered_response"
    ]
    .astype(str)
    .str.strip()
    ==
    ""
).any():

    raise RuntimeError(
        "Empty steered_response values."
    )


if (
    final_df[
        "prompt_template_version"
    ]
    !=
    PROMPT_TEMPLATE_VERSION
).any():

    raise RuntimeError(
        "Mixed prompt templates found."
    )


print(
    "\nFinal counts:"
)


print(

    final_df[
        "objective_id"
    ]
    .value_counts()
    .reindex(
        list(
            OBJECTIVES
        )
    )
)


final_df.to_csv(

    OUTPUT_CSV,

    index=False,
)


# ============================================================
# 28. METADATA
# ============================================================

metadata = {

    "project":
        "AIMES",

    "version":
        VERSION,

    "method":
        METHOD,

    "baseline_name":
        BASELINE_NAME,

    "prompt_template_version":
        PROMPT_TEMPLATE_VERSION,

    "model_key":
        MODEL_KEY,

    "model_id":
        MODEL_ID,

    "model_save_name":
        MODEL_SAVE,

    "created_at":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "shared_prompt_manifest":
        HF_MANIFEST,

    "target_disjoint_only":
        True,

    "objective_counts":
        {
            key:
                int(
                    value
                )

            for key, value
            in EXPECTED_COUNTS.items()
        },

    # AIMES sign-level objectives
    "objectives":
        OBJECTIVES,

    # Explicit prompt intensity targets
    "intensity_profiles":
        INTENSITY_PROFILES,

    "intensity_scale":
        {
            str(key):
                value

            for key, value
            in INTENSITY_SCALE.items()
        },

    "value_definitions":
        VALUE_DEFINITIONS,

    "prompting_method":
        (
            "VALUEFLOW-style discrete intensity-anchor "
            "multi-value prompting"
        ),

    "methodological_note":
        (
            "This baseline adapts the intensity-anchor prompting "
            "paradigm used in VALUEFLOW to the AIMES Moral "
            "Foundations objectives. It is not claimed to be a "
            "verbatim reproduction of the VALUEFLOW prompt "
            "template. Strong +/-2 anchors are used for all "
            "requested dimensions to provide a strong prompt-based "
            "control baseline."
        ),

    "reference":
        {
            "title":
                (
                    "VALUEFLOW: Toward Pluralistic and "
                    "Steerable Value-based Alignment in "
                    "Large Language Models"
                ),

            "authors":
                (
                    "Woojin Kim, Sieun Hyeon, "
                    "Jusang Oh, Jaeyoung Do"
                ),

            "venue":
                "ICML 2026",

            "arxiv":
                "2602.03160",
        },

    "generation": {

        "max_new_tokens":
            MAX_NEW_TOKENS,

        "do_sample":
            DO_SAMPLE,

        "repetition_penalty":
            REPETITION_PENALTY,

        "generation_seed":
            GENERATION_SEED,

        "qwen_enable_thinking":
            (
                QWEN_ENABLE_THINKING

                if MODEL_FAMILY == "qwen3"

                else

                None
            ),
    },

    "note":
        (
            "Prompt steering is layer-independent. Each "
            "target-disjoint prompt/objective pair is generated "
            "once and reused when comparing against layer-specific "
            "activation-steering methods."
        ),
}


with open(

    METADATA_JSON,

    "w",

    encoding=
        "utf-8",

) as f:

    json.dump(

        metadata,

        f,

        indent=
            2,

        ensure_ascii=
            False,
    )


# ============================================================
# 29. UPLOAD TO HF
# ============================================================

operations = [

    CommitOperationAdd(

        path_in_repo=
            HF_OUTPUT_CSV,

        path_or_fileobj=
            str(
                OUTPUT_CSV
            ),
    ),


    CommitOperationAdd(

        path_in_repo=
            HF_METADATA_JSON,

        path_or_fileobj=
            str(
                METADATA_JSON
            ),
    ),
]


commit = hf_api.create_commit(

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    operations=
        operations,

    commit_message=
        (
            "Add VALUEFLOW-style intensity-anchor "
            f"prompt baseline for {MODEL_SAVE}"
        ),

    token=
        HF_TOKEN,
)


# ============================================================
# 30. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "INTENSITY-ANCHOR PROMPT STEERING COMPLETE"
)

print(
    "=" * 100
)


print(
    "Model:",
    MODEL_NAME
)


print(
    "Baseline:",
    BASELINE_NAME
)


print(
    "Template:",
    PROMPT_TEMPLATE_VERSION
)


print(
    "Rows:",
    len(
        final_df
    )
)


print(
    "Generated this run:",
    generated_this_run
)


print(
    "\nHF generation file:"
)

print(
    HF_OUTPUT_CSV
)


print(
    "\nHF metadata file:"
)

print(
    HF_METADATA_JSON
)


print(
    "\nCommit:"
)

print(
    commit
)